# AV2 — Clusterização de investidores para matchmaking

Objetivo: descobrir perfis de investidores sem rótulos e usá-los para desempatar matches com score idêntico. Pipeline: dados da AV1 → feature engineering → feature selection → K-Means → validação → árvore substituta exportada para o backend.

Execute as células em ordem a partir da pasta `entrega-ia/av2`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib

matplotlib.use('Agg')

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd() if (Path.cwd() / 'scripts').exists() else Path('entrega-ia/av2')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from scripts.dados import carregar_bases
from scripts.features import FEATURES_INVESTIDOR, construir_features_investidores
from scripts.modelos import (
    avaliar_estabilidade,
    avaliar_k,
    comparar_algoritmos,
    treinar_modelo,
)
from scripts.selecao import FEATURES_CLUSTERIZACAO, escolher_k, selecionar_features

SEED = 20261006
print('Pasta de trabalho:', RAIZ)

## 1. Dados

O carregador prefere a base preparada da AV1 e valida as colunas necessárias. Se o esquema estiver incompatível, usa a base original e registra o motivo — a AV2 faz a própria preparação.

In [ ]:
startups, investidores, origem = carregar_bases()
print('Startups:', startups.shape, 'Investidores:', investidores.shape)
print('Origem:', json.dumps(origem, ensure_ascii=False, indent=2))

## 2. Feature engineering

Normalização de setores/estágios, moeda, percentual e UF; deduplicação por `id_investidor`; imputação por mediana. Resultado: matriz numérica sem `NaN`.

In [ ]:
X_completa, meta = construir_features_investidores(investidores)
print('Matriz completa:', X_completa.shape)
print('NaN:', int(X_completa.isna().sum().sum()))
display(X_completa.head())

## 3. Feature selection

A clusterização usa apenas o perfil de interesses: 12 features. Ticket, região, experiência e volume de negócios foram descartados por já entrarem no score ou por dominarem a variância sem melhorar a separação.

In [ ]:
X = selecionar_features(X_completa)
print('Features usadas:', list(FEATURES_CLUSTERIZACAO))
print('Descartadas:', [f for f in FEATURES_INVESTIDOR if f not in FEATURES_CLUSTERIZACAO])
display(X.head())

## 4. Escolha de k (silhueta, Calinski-Harabasz e Davies-Bouldin)

In [ ]:
ks = tuple(range(2, 9))
avaliacao = avaliar_k(X, ks, seed=SEED)
display(avaliacao)

fig, eixo = plt.subplots(figsize=(7, 4))
eixo.bar(avaliacao['k'].astype(str), avaliacao['silhueta'], color='#2f6f4f')
eixo.set_title('Silhueta média por k')
eixo.set_xlabel('k')
eixo.set_ylabel('silhueta')
plt.show()

k = escolher_k(avaliacao)
print('k escolhido:', k)

## 5. Treino, estabilidade e comparação de algoritmos

In [ ]:
pipeline, rotulos = treinar_modelo(X, k, seed=SEED)
estabilidade = avaliar_estabilidade(X, k, seed=SEED)
comparacao = comparar_algoritmos(X, k, seed=SEED)
print('Estabilidade (ARI):', estabilidade)
display(comparacao)

## 6. Interpretação dos segmentos

In [ ]:
from collections import Counter

perfis = []
rotulos_serie = pd.Series(rotulos, name='cluster')
for cluster in sorted(set(rotulos)):
    selecionados = meta.loc[rotulos_serie == cluster]
    setores = Counter(
        setor for lista in selecionados['setores_normalizados'] for setor in set(lista)
    )
    perfis.append(
        {
            'cluster': int(cluster),
            'n_investidores': int(len(selecionados)),
            'setores_top': ', '.join(nome for nome, _ in setores.most_common(3)),
        }
    )
perfis = pd.DataFrame(perfis)
display(perfis)

## 7. Exportação

A árvore substituta reproduz os rótulos do K-Means e é gravada em `backend/src/content/segmentos-matchmaking.json`. A fidelidade mínima estimada é 0,85; o valor real obtido está em `evidencias/metricas.json`. A árvore exportada usa 100% dos rótulos; a fidelidade é estimada em 25% não visto.

In [ ]:
metricas = json.loads((RAIZ / 'evidencias' / 'metricas.json').read_text(encoding='utf-8'))
print('Silhueta final:', round(metricas['silhueta_final'], 4))
print('Fidelidade estimada:', round(metricas['fidelidade_arvore'], 3))
print('Arquivos de evidência:')
for caminho in sorted((RAIZ / 'evidencias').rglob('*')):
    if caminho.is_file():
        print(' -', caminho.relative_to(RAIZ))

## Conclusão

O modelo entrega seis perfis interpretáveis e um desempate auditável. Os números são moderados (silhueta 0,18; ARI 0,59), por isso a IA refina a ordenação sem alterar o score calculado pela AV1. Próximo passo: capturar desfechos reais para treinar um modelo supervisionado de propensão.